***Cellule 1 : Préparation des données***

In [4]:
#importation les bibliothéques nécessaires 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [5]:
# Chargement du dataset crée au notebook 05
df_final = pd.read_csv("../resultats/dataset_prediction_liens.csv")

In [10]:
# Sélection des features (on exclut les IDs et la cible)
features_cols = [ 'comp_degree', 'preferential_attachment', 'cluster_id', 'same_location', 'same_industry']
X = df_final[features_cols]
y = df_final['target']

In [11]:
# Split 70% train / 30% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"✅ Données prêtes. Taille test : {len(X_test)} échantillons.")

✅ Données prêtes. Taille test : 600 échantillons.


**Cellule 2 : Entraînement des modèles ML**

In [12]:
# 1. Logistic Regression (Modèle simple/linéaire)
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# 2. Random Forest (Modèle complexe/non-linéaire)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("✅ Entraînement des modèles ML terminé.")

✅ Entraînement des modèles ML terminé.


***Cellule 3 : Évaluation et choix du meilleur modèle***

In [13]:
def calculer_metriques(y_true, y_pred, nom_modele):
    return {
        "Modèle": nom_modele,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Précision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred)
    }

stats = [
    calculer_metriques(y_test, y_pred_lr, "Régression Logistique"),
    calculer_metriques(y_test, y_pred_rf, "Random Forest")
]

df_perf = pd.DataFrame(stats)
print(df_perf.to_string(index=False))

# Sauvegarde pour le rapport final
df_perf.to_csv("../resultats/performance_modeles.csv", index=False)

               Modèle  Accuracy  Précision   Recall  F1-Score
Régression Logistique  0.971667   0.989619 0.953333  0.971138
        Random Forest  0.970000   0.982877 0.956667  0.969595


***Génération des recommandations***

In [14]:
# 1. On filtre pour ne garder QUE les cas où l'employé ne travaille pas dans l'entreprise (target == 0)
df_futur = df_final[df_final['target'] == 0].copy()

In [15]:
# 2. On isole les features à donner au modèle
X_futur = df_futur[features_cols]

In [ ]:
# 3. On calcule la probabilité de recrutement avec la Régression Logistique
# predict_proba renvoie la probabilité d'appartenir à la classe 1 (le lien se crée)
df_futur['probabilite_recrutement'] = lr_model.predict_proba(X_futur)[:, 1]

In [ ]:
# 4. On trie du plus probable au moins probable
df_recommandations = df_futur.sort_values(by='probabilite_recrutement', ascending=False)

In [18]:
# 5. Affichage du Top 10 des recrutements les plus probables
print("🌟 TOP 10 des prédictions de futurs recrutements :")
colonnes_a_afficher = ['employee_id', 'company_name', 'probabilite_recrutement']
print(df_recommandations[colonnes_a_afficher].head(10).to_string(index=False))

🌟 TOP 10 des prédictions de futurs recrutements :
                  employee_id                             company_name  probabilite_recrutement
       keith-freeman-379a1111                                      NaN                 1.000000
        hussam-alaa-09843411a                             at rich bake                 0.979878
  harrison-ainsworth-5b42b951                   blackberry corporation                 0.978791
             gail-fride-dwyer                            self employed                 0.289221
ronnie-caslin-caslin-73b835a2                            self employed                 0.289221
        manuela-dias-b868b29b                                  infosys                 0.164341
      carlton-frank-6bba271b3              national geographic society                 0.158117
           tony-diaz-a9599087 universit de fribourguniversitt freiburg                 0.158117
 iswarjono-iswarjono-5679941a                                   amazon                

In [19]:
# 6. Sauvegarde dans un fichier CSV dédié
chemin_reco = "../resultats/recommandations_futures_carrieres.csv"
df_recommandations[colonnes_a_afficher].to_csv(chemin_reco, index=False)

print(f"\n✅ Fichier de prédictions sauvegardé avec succès : {chemin_reco}")


✅ Fichier de prédictions sauvegardé avec succès : ../resultats/recommandations_futures_carrieres.csv
